# Demographic Table — ICI Cohort (Table 1, main text)

Produces **Table 1**: ICI-treated cohort (known RECIST response) patient and
tumour characteristics, stratified by responder status (RECIST PR/CR vs all
others). This is the primary clinical cohort behind the response/survival
analyses (Figures 3-6).

For the full pan-cancer discovery/validation cohort (Figure 2), see
**Supplementary Table S3** (`11b_demographic_table_full_cohort.ipynb`).

Continuous variables: median (IQR); Wilcoxon rank-sum p-value.  
Categorical variables: n (%); χ² p-value (Fisher's exact for small cells).

Treatment is summarized only at the mechanism level (**ICI Mechanism**:
Anti-PD1 / Anti-PDL1 / Anti-CTLA4 / Combo) — individual drug regimens
aren't broken out as their own row in this main-text table.

The "ICI Cohort" row splits `cohort_broad`'s **Other** category into
**Non-CRC MSI** and **Other MSS** (by `msi_high`) for transparency, since
"Other" otherwise mixes MSI-high and microsatellite-stable patients from
many different tumor types together. SKCM/NSCLC/BLCA/CRC-MSI are shown
unchanged — this split only affects the residual "Other" bucket, and is a
display-level change local to this table (the underlying `cohort_broad`
scheme used elsewhere in the pipeline is untouched). A small residual
**Other (Unknown MSI)** category holds Other-bucket patients whose MSI
status wasn't determined, so no patient silently drops out of the row.

**Input:** `cpi_ready.rds`  
**Output:** `figs/table_1_demographics.docx` (the only output -- this is a
main-text table, so no .csv/.html are generated).


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

library(gtsummary)
library(flextable)

In [ ]:
# ── Load ICI-ready data ───────────────────────────────────────────────────────
# cpi_ready.rds: ICI-treated patients, purity-filtered, CIDER scores computed.
# Restrict to T-cell group (primary analysis) with known RECIST response.
#
# IMPORTANT: cpi_ready.rds stores one row per (sampleId, gp) *and* a duplicated
# copy of every row with cohort_broad overwritten to "ICI-Treated" (added in
# 4_base.ipynb so Figure 3 can facet pooled-vs-per-cancer-type together). The
# gp filter below removes the first duplication; distinct() removes the
# second. Without both, every patient is counted twice and "ICI Cohort"
# would show "ICI-Treated" as if it were a category alongside SKCM/NSCLC/etc.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(gp == "T-cell", !is.na(response)) |>
  distinct(sampleId, .keep_all = TRUE)


In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# tmbPerMb is the raw value; no back-transform needed for display.
# AmberMeanDepth used as WGS sequencing depth proxy.
#
# Variable order: patient demographics (Age, Sex), then the tumour/genomic
# block — Purity, WGS depth, CIDER T-cell score (the paper's primary
# metric), TMB, MSI-H — then cohort/treatment grouping variables (ICI
# Cohort, Prior treatment, ICI Mechanism).
#
# ICI Cohort and ICI Mechanism are ordered by descending count, but with
# their residual catch-all categories pinned to the last rows regardless
# of their count — see fct_pin_last() in helpers/shortcuts.r. Individual
# drug regimens (the raw `treatment` column) are not shown anywhere in
# this table -- only the coarser ICI Mechanism grouping.

demo <- df_raw |>
  mutate(
    Responder             = factor(responder, levels = c("Responder", "Non-Responder")),
    Age                   = as.numeric(age),
    # sex retains whatever casing/abbreviation clinical_update.txt uses
    # (5_models.ipynb defensively matches both "female" and "f" via
    # tolower(), suggesting the raw values aren't a single consistent
    # form). Normalize case-insensitively and output clean, capitalized
    # labels rather than passing the raw column through unchanged.
    Sex                   = factor(
      case_when(
        tolower(sex) %in% c("female", "f") ~ "Female",
        tolower(sex) %in% c("male",   "m") ~ "Male",
        TRUE ~ NA_character_
      ),
      levels = c("Female", "Male")
    ),
    `Purity (%)`          = as.numeric(purity) * 100,  # stored as 0-1 fraction; convert for display
    `WGS depth`           = as.numeric(AmberMeanDepth),
    `CIDER T-cell score`  = as.numeric(cider_score_depth_adj),
    `TMB (mut/Mb)`        = tmbPerMb,
    `MSI-H`               = factor(
      ifelse(msi_high == 1, "Yes", "No"),
      levels = c("Yes", "No")
    ),
    # ICI Cohort: cohort_broad's SKCM/NSCLC/BLCA/CRC-MSI categories are
    # shown as-is (these already don't mix MSI-high and MSS patients the
    # way "Other" does -- CRC-MSI is by construction 100% MSI, and
    # SKCM/NSCLC/BLCA aren't stratified by MSI at all). Only the residual
    # "Other" bucket is split into Non-CRC MSI / Other MSS here, for
    # transparency about the MSI composition it otherwise hides. This is a
    # display-only relabeling local to Table 1 -- cohort_broad itself
    # (helpers/config.r's BROAD_LEVELS, used throughout the rest of the
    # pipeline) is untouched.
    `ICI Cohort`          = fct_pin_last(
      factor(
        case_when(
          cohort_broad != "Other" ~ as.character(cohort_broad),
          msi_high == 1           ~ "Non-CRC MSI",
          msi_high == 0           ~ "Other MSS",
          TRUE                    ~ "Other (Unknown MSI)"
        )
      ),
      c("Non-CRC MSI", "Other MSS", "Other (Unknown MSI)")
    ),
    # pretreated in cpi_ready.rds/ready.rds takes numeric values 1/0 (not
    # text) — confirmed directly against the data. == coerces correctly
    # whether pretreated is stored as numeric or character "1"/"0".
    `Prior treatment`     = factor(
      case_when(
        pretreated == 1 ~ "Yes",
        pretreated == 0 ~ "No",
        TRUE ~ NA_character_
      ),
      levels = c("Yes", "No")
    ),
    `ICI Mechanism`       = fct_pin_last(factor(mechanism), "Multiple mechanism")
  ) |>
  select(
    Responder,
    Age,
    Sex,
    `Purity (%)`,
    `WGS depth`,
    `CIDER T-cell score`,
    `TMB (mut/Mb)`,
    `MSI-H`,
    `ICI Cohort`,
    `Prior treatment`,
    `ICI Mechanism`
  )

In [ ]:
# ── Build Table 1 ─────────────────────────────────────────────────────────────
# Continuous: median (IQR), Wilcoxon rank-sum.
# Categorical: n (%), χ² (Fisher's exact for small cells).

tbl <- demo |>
  tbl_summary(
    by        = Responder,
    statistic = list(
      all_continuous()  ~ "{median} ({p25}–{p75})",
      all_categorical() ~ "{n} ({p}%)"
    ),
    digits    = list(
      all_continuous()  ~ 0,
      # TMB keeps 1 decimal (unlike other continuous variables): the paper
      # uses TMB_HIGH_THRESHOLD = 10 (helpers/config.r) as a clinical
      # cutoff, and rounding to whole numbers would blur values right next
      # to that threshold (e.g. 9.6 -> 10). This override is listed after
      # the all_continuous() rule above, so it takes precedence for this
      # one variable.
      `TMB (mut/Mb)`    ~ 1,
      all_categorical() ~ c(0, 0)
    ),
    missing   = "no"
  ) |>
  add_overall() |>
  add_p(
    test = list(
      all_continuous()  ~ "wilcox.test",
      all_categorical() ~ "chisq.test.no.correct"
    ),
    pvalue_fun = ~style_pvalue(.x, digits = 3)
  ) |>
  # Show N and % of the total in each comparison column's header (but not
  # the Overall column, whose % would always just read 100%), so the split
  # between groups is visible at a glance without a reader doing the math.
  modify_header(
    all_stat_cols(stat_0 = FALSE) ~ "**{level}**, N = {n} ({style_percent(p, digits = 1)}%)"
  ) |>
  bold_labels()

tbl

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
# Table 1 is a main-text table: only a ready-to-use .docx is produced (no
# .csv/.html -- those aren't needed by anything downstream, unlike the
# supplementary tables' .csv, which feeds assemble_supplementary_tables_excel.R).
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

# Word — flextable conversion for direct manuscript use
tbl |>
  as_flex_table() |>
  flextable::save_as_docx(path = paste0(out_dir, "table_1_demographics.docx"))

message("Saved: table_1_demographics.docx")
